In [8]:
df_patients = spark.read.table("Bronze_LH.dbo.bronze_patients")

display(df_patients)

df_admissions = spark.read.table("Bronze_LH.dbo.bronze_admissions")

display(df_admissions)

df_medical_test = spark.read.table("Bronze_LH.dbo.bronze_medicaltests")   

display(df_medical_test)



StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a6419f4b-a2ed-40b9-9034-7be13b8b30c4)

SynapseWidget(Synapse.DataFrame, 75b26c39-64c6-43b6-8e48-47ec353273f7)

SynapseWidget(Synapse.DataFrame, a27df90c-303f-4d52-ac60-c5ecd2eb40c9)

### For incremental data 

In [9]:
from pyspark.sql.functions import count ,when , sum,col,desc,row_number
from pyspark.sql.window import Window

spec = Window.partitionBy("PatientID").orderBy(desc(
    col("LastModifiedDate")
))
df_patients= (
    df_patients.withColumn("rn",row_number().over(spec))
    .filter(col("rn")==1)
    .drop("rn")
)

StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 11, Finished, Available, Finished, False)

In [10]:
print("Total rows:", df_patients.count())

df_patients.groupBy("PatientID") \
    .count() \
    .filter(col("count") > 1) \
    .show()

StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 12, Finished, Available, Finished, False)

Total rows: 992
+---------+-----+
|PatientID|count|
+---------+-----+
+---------+-----+



### Data Cleaning and Profiling (1-> Patients Data)

In [11]:
df_patients.printSchema()

df_patients.columns

StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 13, Finished, Available, Finished, False)

root
 |-- PatientID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- DateOfBirth: date (nullable = true)
 |-- Gender: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- ContactNumber: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)



['PatientID',
 'FirstName',
 'LastName',
 'DateOfBirth',
 'Gender',
 'City',
 'State',
 'ContactNumber',
 'Email',
 'LastModifiedDate']

In [12]:
## Null profiiling
from pyspark.sql.functions import trim , col , when , sum ,desc 

Null_profile_patients =[]
for c in df_patients.columns:
    null_count = df_patients.filter(col(c).isNull()).count()

    Null_profile_patients.append((c,null_count))

Null_profile_patients_df = spark.createDataFrame(Null_profile_patients,["columns","null_count"])

display(Null_profile_patients_df)

## Blank Profiling

Blank_profile_patients = []
for c in df_patients.columns:
    blank_count = df_patients.filter(trim(col(c))=="").count()
    Blank_profile_patients.append((c,blank_count))

Blank_profile_patients_df = spark.createDataFrame(Blank_profile_patients,["columns","blank_count"])

display(Blank_profile_patients_df)

## Profiling Categorical Columns

Categorical_columns = ["City","State","Gender"]

for c in Categorical_columns:
    print(f"\n====={c}=========")

    df_patients.groupBy(c).count().orderBy(desc("count")).show(truncate=False)


    


StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0fff7fa1-49fe-40a3-a35c-b70263cb4b99)

SynapseWidget(Synapse.DataFrame, a1d8e6df-7238-43fb-b4e6-09e3f5d9b15a)


=====City=========
+---------+-----+
|City     |count|
+---------+-----+
|Chennai  |126  |
|Kolkata  |104  |
|Hyderabad|101  |
|Ahmedabad|100  |
|Delhi    |98   |
|Lucknow  |96   |
|Bengaluru|92   |
|Mumbai   |90   |
|Pune     |86   |
|Jaipur   |85   |
|hyderabad|4    |
|pune     |3    |
|ahmedabad|2    |
|bengaluru|2    |
|lucknow  |1    |
|kolkata  |1    |
|jaipur   |1    |
+---------+-----+


=====State=========
+-------------+-----+
|State        |count|
+-------------+-----+
|Maharashtra  |177  |
|Tamil Nadu   |126  |
|Telangana    |105  |
|West Bengal  |103  |
|Gujarat      |102  |
|Delhi        |97   |
|Uttar Pradesh|96   |
|Karnataka    |94   |
|Rajasthan    |83   |
|NULL         |9    |
+-------------+-----+


=====Gender=========
+-------+-----+
|Gender |count|
+-------+-----+
|Female |118  |
|Male   |115  |
|F      |111  |
|female |106  |
|MALE   |104  |
|M      |92   |
|FEMALE |90   |
|male   |87   |
|       |82   |
|Unknown|80   |
|NULL   |7    |
+-------+-----+



In [13]:
##IDENTIFIER columns

Identifier = df_patients.select("PatientID","FirstName","LastName","ContactNumber","Email").limit(10)

display(Identifier)


StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8d1c91ab-4fa7-46ac-be98-6acc9d361cf8)

In [14]:
## duplicate

dup = df_patients.groupBy(df_patients.columns).count().filter(col("count")>1)

display(dup)


df_patients_clean = df_patients.dropDuplicates()

print("before :",df_patients.count())
print("after :",df_patients_clean.count())

StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 049736af-6b3d-4d91-b711-498f47a26ec1)

before : 992
after : 992


In [15]:
silver_patients= df_patients_clean

silver_patients.printSchema()

StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 17, Finished, Available, Finished, False)

root
 |-- PatientID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- DateOfBirth: date (nullable = true)
 |-- Gender: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- ContactNumber: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)



In [16]:

## string columns 
string_columns = ["FirstName","LastName","DateOfBirth","Gender","City","State","ContactNumber","Email"]

for c in string_columns:
    silver_patients = silver_patients.withColumn(c,trim(col(c)))

display(silver_patients)


StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d1fe1eb4-586d-45d4-b496-43f33b9eb70b)

In [17]:
## City and state
from pyspark.sql.functions import lower , when ,initcap

silver_patients = silver_patients.withColumn("city", lower(trim(col("city"))))

silver_patients= silver_patients.withColumn("city",
when( col("city").isin("Delhi","new_delhi"),"Delhi")
.when(col("city").isin("bangalore", "bengaluru"),"Bengaluru")
.otherwise(initcap(col("city")))

)

silver_patients.groupBy("city").count().show(truncate=False)


silver_patients=silver_patients.withColumn("state",initcap(trim(col("state"))))

silver_patients.groupBy("state").count().show(truncate=False)


StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 19, Finished, Available, Finished, False)

+---------+-----+
|city     |count|
+---------+-----+
|Chennai  |126  |
|Lucknow  |97   |
|Mumbai   |90   |
|Ahmedabad|102  |
|Kolkata  |105  |
|Pune     |89   |
|Delhi    |98   |
|Bengaluru|94   |
|Hyderabad|105  |
|Jaipur   |86   |
+---------+-----+

+-------------+-----+
|state        |count|
+-------------+-----+
|Karnataka    |94   |
|Tamil Nadu   |126  |
|NULL         |9    |
|Gujarat      |102  |
|Delhi        |97   |
|Rajasthan    |83   |
|Maharashtra  |177  |
|West Bengal  |103  |
|Telangana    |105  |
|Uttar Pradesh|96   |
+-------------+-----+



In [18]:
## Gender
from pyspark.sql.functions import upper , trim , col , when

silver_patients=silver_patients.withColumn("Gender",
when(upper(trim(col("Gender")))=="FEMALE","Female")
.when(upper(trim(col("Gender")))=="MALE","Male")
.when(upper(trim(col("Gender")))=="F","Female")
.when(upper(trim(col("Gender")))=="M","Male")
.otherwise(None)

)
silver_patients.groupBy("Gender").count().show(truncate=False)

StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 20, Finished, Available, Finished, False)

+------+-----+
|Gender|count|
+------+-----+
|NULL  |169  |
|Female|425  |
|Male  |398  |
+------+-----+



In [19]:
##Emai;
from pyspark.sql.functions import upper , trim , col , when

silver_patients = silver_patients.withColumn("Email", lower(trim(col("Email"))))

email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

silver_patients=silver_patients.withColumn("Email_Status",
when(col("Email").rlike(email_pattern),"VALID"
).otherwise("VALID"))

silver_patients.filter(
    col("Email_Status") != "VALID"
).select(
    "PatientID",
    "Email",
    "Email_Status"
).show(truncate=False)

StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 21, Finished, Available, Finished, False)

+---------+-----+------------+
|PatientID|Email|Email_Status|
+---------+-----+------------+
+---------+-----+------------+



In [20]:
#phone number
from pyspark.sql.functions import upper , trim , col , when, regexp_extract,regexp_replace , length

silver_patients=silver_patients.withColumn("ContactNumber",regexp_replace(col("ContactNumber"), r"[^0-9+]", "")
)

silver_patients.groupBy(
    length("ContactNumber").alias("number_length")
).count().orderBy("number_length").show()

silver_patients = silver_patients.withColumn(
    "phone_status",
    when(
        col("ContactNumber").rlike(r"^[6-9][0-9]{9}$"),
        "VALID"
    ).otherwise("INVALID")
)

silver_patients.filter(col("phone_status")!="VALID").select("PatientID","ContactNumber","phone_status").show(truncate=False)

StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 22, Finished, Available, Finished, False)

+-------------+-----+
|number_length|count|
+-------------+-----+
|            5|   16|
|           10|  976|
+-------------+-----+

+---------+-------------+------------+
|PatientID|ContactNumber|phone_status|
+---------+-------------+------------+
|59       |98765        |INVALID     |
|118      |98765        |INVALID     |
|177      |98765        |INVALID     |
|236      |98765        |INVALID     |
|295      |98765        |INVALID     |
|354      |98765        |INVALID     |
|413      |98765        |INVALID     |
|472      |98765        |INVALID     |
|531      |98765        |INVALID     |
|590      |98765        |INVALID     |
|649      |98765        |INVALID     |
|708      |98765        |INVALID     |
|767      |98765        |INVALID     |
|826      |98765        |INVALID     |
|885      |98765        |INVALID     |
|944      |98765        |INVALID     |
+---------+-------------+------------+



In [21]:
## DOB

from pyspark.sql.functions import to_date, coalesce

silver_patients = silver_patients.withColumn(
    "DateOfBirth",
    coalesce(
        to_date(col("DateOfBirth"), "yyyy-MM-dd"),
        to_date(col("DateOfBirth"), "dd/MM/yyyy"),
        to_date(col("DateOfBirth"), "yyyy/MM/dd"),
        to_date(col("DateOfBirth"), "dd-MM-yyyy")
    )
)

silver_patients.select(
    "PatientID","DateOfBirth"
).show(truncate=False)



StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 23, Finished, Available, Finished, False)

+---------+-----------+
|PatientID|DateOfBirth|
+---------+-----------+
|1        |1933-12-09 |
|2        |1933-09-07 |
|3        |1943-02-13 |
|4        |2009-06-19 |
|5        |1956-11-09 |
|6        |1970-07-09 |
|7        |2004-07-19 |
|8        |1978-10-15 |
|9        |2022-05-17 |
|10       |1992-01-04 |
|11       |2000-03-09 |
|12       |1938-06-01 |
|13       |1965-11-16 |
|14       |1984-07-15 |
|15       |1989-04-28 |
|16       |1937-03-13 |
|17       |1999-01-24 |
|18       |2016-04-13 |
|19       |1956-11-23 |
|20       |1994-05-05 |
+---------+-----------+
only showing top 20 rows



In [22]:
#checking dups wityh buiseness id
from pyspark.sql.functions import count

duplicate_patient_ids = (
    silver_patients
    .groupBy("PatientID")
    .agg(count("*").alias("count"))
    .filter(col("count") > 1)
)

display(duplicate_patient_ids)


StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 25edc0e7-ce80-41df-aed6-0112b6236b02)

In [23]:
## first name and last name  and DOB status

from pyspark.sql.functions import current_date

silver_patients=silver_patients.withColumn("FirstName_Status",
     
     when(col("FirstName").isNull() |
     (trim(col("FirstName"))==""),"INVALID"
     ).otherwise("VALID"))

silver_patients=silver_patients.withColumn("LastName_Status",
     
     when(col("LastName").isNull() |
     (trim(col("LastName"))==""),"INVALID"
     ).otherwise("VALID"))


silver_patients = silver_patients.withColumn("DOB_Status",
    
     when(col("DateOfBirth").isNull() & (col("DateOfBirth") > current_date()),"INVALID"
    ).otherwise("VALID")
)

silver_patients= silver_patients.withColumn("Gender_Status",
    when(col("Gender").isNull(),"INVALID"
    ).otherwise("VALID")
)


StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 25, Finished, Available, Finished, False)

In [24]:
#Handling State Null Value

silver_patients=silver_patients.withColumn(
    "State",
    when(
        col("state").isNull() & (col("city")=="Mumbai"),"Maharashtra"
    )
    .when(
        col("state").isNull() & (col("city")=="Pune"),"Maharashtra"
    )
    .when(
        col("state").isNull() & (col("city")=="Jaipur"),"Rajasthan"
    )
    .when(col("state").isNull() & (col("city")=="Kolkata"),"West Bengal"
    )
    .when(col("state").isNull() & (col("city")=="Lucknow"),"Uttar Pradesh"
    )
    .when(
        col("State").isNull() & (col("City") == "Delhi"),"Delhi"
    )
    .when(
        col("State").isNull() & (col("City") == "Ahmedabad"),"Gujarat"
    )
     .when(
        col("State").isNull() & (col("City") == "Bengaluru"),"Karnataka"
    )
    .when(
        col("State").isNull() & (col("City") == "Hyderabad"),"Telangana"
    ).otherwise(col("State"))
)

silver_patients.filter(col("state").isNull()).select(
    "PatientID","city","State"
).show(truncate=False)

display(silver_patients)



StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 26, Finished, Available, Finished, False)

+---------+----+-----+
|PatientID|city|State|
+---------+----+-----+
+---------+----+-----+



SynapseWidget(Synapse.DataFrame, b3f75897-44d5-4e57-9262-a404f39fd29a)

In [25]:
from pyspark.sql.functions import concat_ws, when, lit

silver_patients=silver_patients.withColumn(
    "DataQualityReason",
    concat_ws(
        ",",
        when(
            col("FirstName_Status")=="INVALID",
            lit("MISSING FIRST NAME")
        ),
        when(
            col("LastName_Status")=="INVALID",
            lit("MISSING LAST NAME")
        ),
        when(
            col("Email_Status")!="VALID",
            lit("INVALID EMAIL")
        ),
        when(
            col("phone_status") != "VALID",
            lit("INVALID_PHONE")
        ),
        when(
            col("DOB_Status") != "VALID",
            lit("INVALID_DOB")
        ),
        
        when(
            col("Gender_Status") != "VALID",
            lit("INVALID_GENDER")
        )

    )
)

silver_patients=silver_patients.withColumn(
    "DataQualityStatus",
    when(
        col("DataQualityReason")=="","VALID"
    ).otherwise("REVIEW")
)

silver_patients.groupBy(
    "DataQualityStatus"
).count().show()



StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 27, Finished, Available, Finished, False)

+-----------------+-----+
|DataQualityStatus|count|
+-----------------+-----+
|           REVIEW|  207|
|            VALID|  785|
+-----------------+-----+



In [30]:
silver_patients_valid = silver_patients.filter(
    col("DataQualityStatus")=="VALID"
)

silver_patients_review = silver_patients.filter(
    col("DataQualityStatus") == "REVIEW"
)

silver_patients_valid.filter(col("PatientID")==1).show()

StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 32, Finished, Available, Finished, False)

+---------+---------+--------+-----------+------+------+-----------+-------------+--------------------+--------------------+------------+------------+----------------+---------------+----------+-------------+-----------------+-----------------+
|PatientID|FirstName|LastName|DateOfBirth|Gender|  city|      State|ContactNumber|               Email|    LastModifiedDate|Email_Status|phone_status|FirstName_Status|LastName_Status|DOB_Status|Gender_Status|DataQualityReason|DataQualityStatus|
+---------+---------+--------+-----------+------+------+-----------+-------------+--------------------+--------------------+------------+------------+----------------+---------------+----------+-------------+-----------------+-----------------+
|        1|    Sneha|   Verma| 1933-12-09|Female|Mumbai|Maharashtra|   9218196001|sneha.verma1@exam...|2026-08-24 23:59:...|       VALID|       VALID|           VALID|          VALID|     VALID|        VALID|                 |            VALID|
+---------+---------

In [33]:
silver_patients_valid.createOrReplaceTempView(
    "new_valid_patients"
)


spark.sql("""

MERGE INTO silver_patients AS target
USING new_valid_patients AS source
ON target.PatientID = source.PatientID

WHEN MATCHED THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *

""")

StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 35, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [35]:
silver_patients_review.createOrReplaceTempView(
    "new_review_patients"
)

spark.sql("""

MERGE INTO silver_patients_review as target
USING new_review_patients AS source
ON target.PatientID = source.PatientID

WHEN MATCHED THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *





""")

StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 37, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [34]:

spark.table("silver_patients") \
    .filter(col("PatientID") == 1) \
    .show(truncate=False)

StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 36, Finished, Available, Finished, False)

+---------+---------+--------+-----------+------+------+-----------+-------------+------------------------+----------------------+------------+------------+----------------+---------------+----------+-------------+-----------------+-----------------+
|PatientID|FirstName|LastName|DateOfBirth|Gender|city  |State      |ContactNumber|Email                   |LastModifiedDate      |Email_Status|phone_status|FirstName_Status|LastName_Status|DOB_Status|Gender_Status|DataQualityReason|DataQualityStatus|
+---------+---------+--------+-----------+------+------+-----------+-------------+------------------------+----------------------+------------+------------+----------------+---------------+----------+-------------+-----------------+-----------------+
|1        |Sneha    |Verma   |1933-12-09 |Female|Mumbai|Maharashtra|9218196001   |sneha.verma1@example.com|2026-08-24 23:59:03.58|VALID       |VALID       |VALID           |VALID          |VALID     |VALID        |                 |VALID          

In [29]:
spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 31, Finished, Available, Finished, False)

+----------------------------------+-------------------------+-----------+
|namespace                         |tableName                |isTemporary|
+----------------------------------+-------------------------+-----------+
|SmartHealth_Insights.SILVER_LH.dbo|silver_admissions_review |false      |
|SmartHealth_Insights.SILVER_LH.dbo|silver_admissions_valid  |false      |
|SmartHealth_Insights.SILVER_LH.dbo|silver_medicaltest_review|false      |
|SmartHealth_Insights.SILVER_LH.dbo|silver_medicaltest_valid |false      |
|SmartHealth_Insights.SILVER_LH.dbo|silver_patients          |false      |
|SmartHealth_Insights.SILVER_LH.dbo|silver_patients_review   |false      |
+----------------------------------+-------------------------+-----------+



In [27]:
### OLD INITIAL LOAD LOGIC - NOT USED FOR INCREMENTAL LOAD

##silver_patients_valid.write.format("delta").mode("overwrite")\
##.saveAsTable("silver_patients")



StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 29, Finished, Available, Finished, False)

In [28]:
### OLD INITIAL LOAD LOGIC - NOT USED FOR INCREMENTAL LOAD
##silver_patients_review.write \
 ##   .format("delta") \
  ##  .mode("overwrite") \
   ## .saveAsTable("silver_patients_review")

StatementMeta(, d691fdfb-bf2a-4217-a5d5-d0bfca98f879, 30, Finished, Available, Finished, False)